In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")


In [4]:
df=pd.read_csv(r"C:\course\datascience udemy\Complete-Data-Science-With-Machine-Learning-And-NLP-2024\11-Random Forest\Projects\Regression\data\cardekho_imputated.csv", index_col=[0])
df

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19537,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
19540,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
19541,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
19542,Mahindra XUV500,Mahindra,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


In [5]:
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [6]:
df.drop('car_name', axis=1, inplace=True)
df.drop('brand', axis=1, inplace=True)

In [32]:
df['model'].unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [97]:
#getting all different type of features
num_features=df.select_dtypes(exclude=[object]).columns
print(num_features)
cat_features=df.select_dtypes(include=[object]).columns
print(cat_features)
discrete_features=[feature for feature in num_features if len(df[feature].unique())<=25]
print('Num of Discrete Features :',len(discrete_features))
continuous_features=[feature for feature in num_features if feature not in discrete_features]
print('Num of Continuous Features :',len(continuous_features))

Index(['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats',
       'selling_price'],
      dtype='object')
Index(['model', 'seller_type', 'fuel_type', 'transmission_type'], dtype='object')
Num of Discrete Features : 2
Num of Continuous Features : 5


In [130]:
from sklearn.model_selection import train_test_split
X=df.drop('selling_price',axis=1)
y=df['selling_price']

num_features=X.select_dtypes(exclude=[object]).columns
y

0         120000
1         550000
2         215000
3         226000
4         570000
          ...   
19537     250000
19540     925000
19541     425000
19542    1225000
19543    1200000
Name: selling_price, Length: 15411, dtype: int64

## Feature Encoding and Scaling
**One Hot Encoding for Columns which had lesser unique values and not ordinal**
* One hot encoding is a process by which categorical variables are converted into a form that could be provided to ML algorithms to do a better job in prediction.

In [131]:
#print lenth of all catagorical features    


print(X.select_dtypes(include=['object']).nunique())


model                120
seller_type            3
fuel_type              5
transmission_type      2
dtype: int64


In [132]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,StandardScaler

label_encoder = LabelEncoder()

one_hot_encoder = OneHotEncoder()

X['model'] = label_encoder.fit_transform(X['model'])
X = pd.get_dummies(X, columns=['seller_type',"fuel_type","transmission_type"], drop_first=True)

X = X.astype(int) 



features_to_scale = [col for col in num_features if col != 'selling_price']

# Apply StandardScaler only to selected features
standard_scaler = StandardScaler()
X[features_to_scale] = standard_scaler.fit_transform(X[features_to_scale])


In [133]:
X.head()

,model,vehicle_age,km_driven,mileage,engine,max_power,seats,seller_type_Individual,seller_type_Trustmark Dealer,fuel_type_Diesel,fuel_type_Electric,fuel_type_LPG,fuel_type_Petrol,transmission_type_Manual
0,7,0.983562,1.247335,-0.059763,-1.324259,-1.262112,-0.403022,1,0,0,0,0,1,1
1,54,-0.343933,-0.690016,-0.298306,-0.554718,-0.424505,-0.403022,1,0,0,0,0,1,1
2,118,1.647309,0.084924,-0.536848,-0.554718,-0.471038,-0.403022,1,0,0,0,0,1,1
3,7,0.983562,-0.360667,0.178779,-0.936610,-0.773508,-0.403022,1,0,0,0,0,1,1
4,38,-0.012060,-0.496281,0.655864,0.022918,-0.052235,-0.403022,0,0,1,0,0,0,1


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error



((12328, 14), (3083, 14))

In [157]:
#create a function to evaluate model
def evaluate(y_test,y_pred):

   
    r2=r2_score(y_test,y_pred)
    mae=mean_absolute_error(y_test,y_pred)
    mse=mean_squared_error(y_test,y_pred)
    return r2,mae,mse


In [158]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
   
}

In [160]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
for name, model in models.items():
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

Lasso
Model performance for Training set
- Root Mean Squared Error: 554150.1795
- Mean Absolute Error: 268579.0372
- R2 Score: 0.6214
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502625.5677
- Mean Absolute Error: 280214.6883
- R2 Score: 0.6644


Lasso
Model performance for Training set
- Root Mean Squared Error: 554150.1795
- Mean Absolute Error: 268579.0372
- R2 Score: 0.6214
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502625.5677
- Mean Absolute Error: 280214.6883
- R2 Score: 0.6644


Lasso
Model performance for Training set
- Root Mean Squared Error: 554150.1795
- Mean Absolute Error: 268579.0372
- R2 Score: 0.6214
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502625.5677
- Mean Absolute Error: 280214.6883
- R2 Score: 0.6644


Lasso
Model performance for Training set
- Root Mean Squared Error: 554150.1795
- Mean Absolute Error: 268579.0372


In [161]:
knn_params = {"n_neighbors": [2, 3, 10, 20, 40, 50]}
rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}



In [181]:
from sklearn.metrics import r2_score

randomcv_models = [
    ('KNN', KNeighborsRegressor(), knn_params),
    ('RF', RandomForestRegressor(), rf_params)
]

for name, model, params in randomcv_models:
    print(f"\nRunning RandomizedSearchCV for {name}...")
    randomcv = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=100,
        cv=5,
        verbose=2,
        random_state=42,
        n_jobs=-1
    )
    
    randomcv.fit(X_train, y_train)
    y_pred = randomcv.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    
    print(f"{name} Best Parameters:", randomcv.best_params_)
    print(f"{name} Best CV Score: {randomcv.best_score_}")
   



Running RandomizedSearchCV for KNN...
Fitting 5 folds for each of 6 candidates, totalling 30 fits
KNN Best Parameters: {'n_neighbors': 2}
KNN Best CV Score: 0.8503218171513629

Running RandomizedSearchCV for RF...
Fitting 5 folds for each of 100 candidates, totalling 500 fits
RF Best Parameters: {'n_estimators': 500, 'min_samples_split': 2, 'max_features': 7, 'max_depth': 15}
RF Best CV Score: 0.8861758245518523


In [ ]:
## Retraining the models with best parameters
models = {
    "Random Forest Regressor": RandomForestRegressor(n_estimators=500, min_samples_split=2, max_features=7, max_depth=15, 
                                                     n_jobs=-1),
     "K-Neighbors Regressor": KNeighborsRegressor(n_neighbors=2, n_jobs=-1)
    
}
for name, model in models.items():
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

Random Forest Regressor
Model performance for Training set
- Root Mean Squared Error: 144961.6549
- Mean Absolute Error: 55113.7169
- R2 Score: 0.9741
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 215016.9058
- Mean Absolute Error: 98331.0264
- R2 Score: 0.9386


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 177425.0924
- Mean Absolute Error: 60891.4260
- R2 Score: 0.9612
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 256372.9378
- Mean Absolute Error: 110842.2397
- R2 Score: 0.9127




In [182]:
from cuml.ensemble import RandomForestRegressor as cuRF
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score
import cuml

# cuML uses cupy arrays for GPU acceleration
from cuml.preprocessing.model_selection import train_test_split
import cupy as cp

X_train_cp = cp.asarray(X_train)
X_test_cp = cp.asarray(X_test)
y_train_cp = cp.asarray(y_train)
y_test_cp = cp.asarray(y_test)

params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 20],
    'max_features': ['auto', 'sqrt'],
    'min_samples_split': [2, 5, 10],
}

model = cuRF()

# cuML doesn't yet support sklearn's RandomizedSearchCV out-of-the-box
# So you can loop manually or use Optuna for GPU-based hyperparameter tuning
model.fit(X_train_cp, y_train_cp)
y_pred = model.predict(X_test_cp)
r2 = cuml.metrics.r2_score(y_test_cp, y_pred)

print("R2 Score:", r2)


ModuleNotFoundError: No module named 'cuml'